# PrecisionSkin – Acne Detector Training + Data-Size Comparison

You run this yourself. **Run the cells in order** (Run All works). Total time about 40–80 minutes on your RTX 4060.

**What it does**
1. Copies your merged acne dataset (`Acne_Boxing/acne_dataset_v3`) to WSL's own disk (reading thousands of files over the Windows drive can fail).
2. Trains the same detector on **25 %, 50 % and 100 %** of your training photos.
3. Scores every model on your **167 hand-boxed test photos** (never trained on, never used to choose anything) with the same code, next to **v1** (the detector in the app).
4. Tells you in plain words: does more data still help (is more boxing worth it?) and does the new detector beat v1?
5. Exports the best model to ONNX.

| Cell | What it does |
|---|---|
| 1 | Packages and GPU check |
| 2 | **Settings** (the only cell you may edit) |
| 3 | Copies and checks the dataset |
| 4 | Trains the three sizes (resumable; takes the longest) |
| 5 | Scores everything on validation and test, plus v1 |
| 6 | Results table and the data-size curve |
| 7 | Plain-language verdict |
| 8 | ONNX export of the best model |

**Read this:** the test photos are the 167 you boxed from scratch. A score is only comparable inside this notebook (same photos, same scoring code). Do not compare it with numbers from papers or from the old public test set.

## CELL 1 – Packages and GPU

In [ ]:
# CELL 1 – Packages and GPU
import importlib, importlib.util, sys
need = ["ultralytics", "cv2", "pandas", "numpy", "yaml", "matplotlib", "torch"]
missing = [m for m in need if importlib.util.find_spec(m) is None]
assert not missing, "Missing packages: %s  (in .venv_det: pip install ultralytics opencv-python-headless pandas pyyaml matplotlib)" % missing
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY (very slow)")

## CELL 2 – Settings (edit only this cell)

In [ ]:
# CELL 2 – Settings
from pathlib import Path
SMOKE = False                      # True = a 1-epoch plumbing test only (never use its numbers)

HOME     = Path.home() / "PrecisionSkinV1"
DATASET  = Path("/mnt/c/Users/Bautista/Downloads/PrecisionSkin/Acne_Boxing/acne_dataset_v4")      # made by merge_acne_dataset.py (v4 = clean set: no machine-drafted non-facial photos, validation boxed from scratch only)
OUT      = HOME / "acne_detector_v4"                                                                # everything this notebook writes
V1_MODEL = HOME / "detector_work" / "runs" / "detector_v1_yolov8n_seed42" / "weights" / "best.pt"  # the detector in the app (3 classes; acne = class 0)
V1_APP_CONF = 0.20                 # acne threshold used in the app

MODEL    = "yolov8s.pt"            # bigger than the app's yolov8n; the result tells you if it is worth it
IMGSZ    = 800
EPOCHS   = 100
PATIENCE = 30
BATCH    = 8
SEEDS    = [42, 7, 123]            # FINAL: three runs with different seeds; report the average and the spread (about 70 minutes in total)
FRACTIONS = [1.00]     # share of the training photos used
HELPER_DIR = [Path("/mnt/c/Users/Bautista/Downloads"), Path("/mnt/c/Users/Bautista/source/repos/SkinDevApp")]   # where hyperpig_autolabel.py (scoring code) is

if SMOKE:
    OUT = HOME / "acne_detector_SMOKE"; EPOCHS = 1; IMGSZ = 320; BATCH = 8; FRACTIONS = [0.25]; MODEL = "yolov8n.pt"
OUT.mkdir(parents=True, exist_ok=True)
for d in HELPER_DIR:
    if (d / "hyperpig_autolabel.py").exists():
        sys.path.insert(0, str(d)); break
else:
    raise FileNotFoundError("hyperpig_autolabel.py not found (it holds the scoring code); keep it in Downloads")
import hyperpig_autolabel as H
print("dataset:", DATASET); print("outputs:", OUT); print("model:", MODEL, "| imgsz", IMGSZ, "| epochs", EPOCHS, "| fractions", FRACTIONS)

## CELL 3 – Copy and check the dataset
Test photos stay in `test/`. Nothing in the original dataset is changed.

In [ ]:
# CELL 3 – Dataset
import shutil, random, yaml, pandas as pd
LOCAL = OUT / "dataset"
def counts(root):
    return {sp: (len(list((root / sp / "images").glob("*"))), sum(len([l for l in f.read_text().splitlines() if l.strip()]) for f in (root / sp / "labels").glob("*.txt"))) for sp in ("train", "val", "test")}
assert (DATASET / "data.yaml").exists(), "Run merge_acne_dataset.py first (acne_dataset_v4 not found)"
src = counts(DATASET)
if not LOCAL.exists() or counts(LOCAL) != src:
    shutil.rmtree(LOCAL, ignore_errors=True); shutil.copytree(DATASET, LOCAL, ignore=shutil.ignore_patterns("*.csv", "*.md"))
loc = counts(LOCAL)
print(pd.DataFrame({"photos": {k: v[0] for k, v in loc.items()}, "boxes": {k: v[1] for k, v in loc.items()}}))
assert loc == src, "copy differs from the source"
tr_names = {p.stem for p in (LOCAL / "train" / "images").glob("*")} | {p.stem for p in (LOCAL / "val" / "images").glob("*")}
te_names = {p.stem for p in (LOCAL / "test" / "images").glob("*")}
assert not (tr_names & te_names), "a photo is in both train/val and test!"
print("test photos are separate from training: OK |", len(te_names), "test photos")
leak = DATASET / "leak_check.csv"
if leak.exists(): print("leak check (looks-alike test/train pairs, already removed from training):", len(pd.read_csv(leak)))

## CELL 4 – Train 25 %, 50 % and 100 % of the training photos
Long. If the kernel stops, run this cell again: finished sizes are skipped and an unfinished one resumes.

In [ ]:
# CELL 4 – Training
from ultralytics import YOLO
all_train = sorted(p for p in (LOCAL / "train" / "images").glob("*"))
runs = {}
for seed in SEEDS:
    for frac in FRACTIONS:
        pct = int(round(frac * 100)); name = "acne_f%d_s%d" % (pct, seed)
        rng = random.Random(seed); sub = all_train[:]; rng.shuffle(sub); sub = sorted(sub[: max(10, int(round(frac * len(sub))))])
        lst = OUT / ("train_%s.txt" % name); lst.write_text("\n".join(str(p) for p in sub) + "\n")
        yml = OUT / ("data_%s.yaml" % name)
        yaml.safe_dump({"path": str(LOCAL), "train": str(lst), "val": "val/images", "test": "test/images", "names": {0: "acne"}}, open(yml, "w"))
        run = OUT / "runs" / name; best = run / "weights" / "best.pt"; done = run / "DONE"
        runs[(frac, seed)] = dict(name=name, best=best, n_train=len(sub))
        if done.exists():
            print("[%s] already trained (%d photos) -> skipped" % (name, len(sub))); continue
        print("=" * 30, name, "| training photos:", len(sub))
        last = run / "weights" / "last.pt"
        if last.exists():
            YOLO(str(last)).train(resume=True)
        else:
            YOLO(MODEL).train(data=str(yml), imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, device=0 if torch.cuda.is_available() else "cpu", workers=2,
                              seed=seed, cos_lr=True, close_mosaic=10, project=str(OUT / "runs"), name=name, exist_ok=True,
                              fliplr=0.5, flipud=0.0, degrees=10, scale=0.5, hsv_h=0.015, hsv_s=0.5, hsv_v=0.35)
        assert best.exists(); done.write_text("ok")
print("\ntrained:", {k: v["name"] for k, v in runs.items()})

## CELL 5 – Score everything the same way
For each new model the confidence used for precision/recall is the one with the best F1 **on validation** (not on test). v1 uses the app's 0.20.
Test = your 167 hand-boxed photos. Scores: mAP@0.5 and mAP@0.5-0.95 (standard), precision, recall, F1 at IoU 0.5.

In [ ]:
# CELL 5 – Scoring
rows = []
def add(label, weights, imgsz, class_id, fixed_conf, n_train, frac, seed):
    sv = H.score_model(weights, imgsz, LOCAL / "val", class_id)
    st = H.score_model(weights, imgsz, LOCAL / "test", class_id)
    conf = fixed_conf if fixed_conf is not None else H.best_f1_conf(sv)
    p, r, f = H.at_conf(st, conf)
    rows.append({"model": label, "train photos": n_train, "fraction": frac, "seed": seed, "val mAP50": round(sv["AP50"], 3), "TEST mAP50": round(st["AP50"], 3),
                 "TEST mAP50-95": round(st["AP50_95"], 3), "conf used": round(conf, 3), "TEST precision": round(p, 3), "TEST recall": round(r, 3), "TEST F1": round(f, 3)})
    print("scored", label)
if V1_MODEL.exists():
    add("v1 (in the app, yolov8n 640)", V1_MODEL, 640, 0, V1_APP_CONF, None, None, None)
else:
    print("v1 model not found at", V1_MODEL, "-> comparison with v1 skipped")
for (frac, seed), r in runs.items():
    add("new %s %d%% of training photos" % (MODEL.replace(".pt", ""), int(round(frac * 100))), r["best"], IMGSZ, None, None, r["n_train"], frac, seed)
res = pd.DataFrame(rows)
res.to_csv(OUT / "results.csv", index=False)
res

## CELL 6 – Results and the data-size curve

In [ ]:
# CELL 6 – Table + curve
import matplotlib.pyplot as plt
display(res)
new = res[res.fraction.notna()].groupby("fraction", as_index=False).agg({"train photos": "mean", "TEST mAP50": "mean", "TEST F1": "mean", "TEST recall": "mean", "TEST precision": "mean"})
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(new["train photos"], new["TEST mAP50"], "o-", label="new detector (mAP@0.5 on your test photos)")
v1 = res[res.fraction.isna()]
if len(v1): ax.axhline(float(v1["TEST mAP50"].iloc[0]), color="red", ls="--", label="v1 in the app")
ax.set_xlabel("training photos used"); ax.set_ylabel("test mAP@0.5"); ax.set_title("Does more data help?"); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig(OUT / "data_size_curve.png", dpi=130); plt.show()

## CELL 7 – Plain-language verdict

In [ ]:
# CELL 7 – Verdict
new = res[res.fraction.notna()].sort_values("fraction")
FULL = float(new.fraction.max())                                  # normally 1.0
def val(frac, col="TEST mAP50"): return float(new[new.fraction == frac][col].mean())
full = val(FULL)
lower = sorted(set(new.fraction) - {FULL})
half = val(lower[-1]) if lower else None                          # the next smaller size (normally 50 %)
print("=" * 70)
print("New detector, %d%% of the training photos (%d): mAP@0.5 = %.3f | precision %.2f | recall %.2f | F1 %.2f" % (
    int(round(FULL * 100)), new[new.fraction == FULL]["train photos"].iloc[0], full, val(FULL, "TEST precision"), val(FULL, "TEST recall"), val(FULL, "TEST F1")))
if half is not None:
    gain = full - half
    print("Going from %d%% to %d%% of the data changed the score by %+.3f mAP@0.5." % (int(round(lower[-1] * 100)), int(round(FULL * 100)), gain))
    if gain >= 0.03: print("-> More data STILL HELPS clearly: boxing more acne photos is worth it.")
    elif gain >= 0.01: print("-> A small gain: more photos help a little; cleaner boxes may matter more.")
    else: print("-> The curve is flat: more photos of the same kind will not help much. Improve label quality or the model instead.")
if len(v1):
    d = full - float(v1["TEST mAP50"].iloc[0])
    print("v1 (in the app): mAP@0.5 = %.3f  ->  new minus v1 = %+.3f" % (float(v1["TEST mAP50"].iloc[0]), d))
    print("-> The new detector is %s v1 on YOUR test photos." % ("BETTER than" if d > 0.02 else "about the same as" if d > -0.02 else "WORSE than"))
    print("   Caveat: v1 may have seen look-alike copies of a few of your test photos in its public training data (this slightly favours v1).")
print("Noise warning: 167 test photos and one seed. Differences under about 0.03 are within noise; use SEEDS=[42,7,123] before you write a conclusion.")
print("=" * 70)

## CELL 8 – Export the best model to ONNX

In [ ]:
# CELL 8 – ONNX export (the model trained on 100 % of the photos)
full_runs = [r for (f, s), r in runs.items() if f == max(FRACTIONS)]
best_w = full_runs[0]["best"]
m = YOLO(str(best_w)); onnx = m.export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True, dynamic=False)
import hashlib, json
sha = hashlib.sha256(open(onnx, "rb").read()).hexdigest()
(OUT / "export_info.json").write_text(json.dumps({"onnx": str(onnx), "sha256": sha, "imgsz": IMGSZ, "classes": ["acne"], "model": MODEL, "dataset": str(DATASET)}, indent=2))
print("ONNX:", onnx); print("sha256:", sha)
print("Not in the app yet: first read the verdict above. The app uses a 3-class detector; this one is acne only.")